|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>Sharing blocks<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: build automatic prefix caching<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import hashlib
from collections import OrderedDict

import numpy as np

rng = np.random.default_rng(0)

Build automatic prefix caching.

You need no tensors. All of the interesting work is in the bookkeeping. The
bookkeeping holds one trap. If you fall into the trap, the engine gives you
fluent, confident and wrong output.

In [ ]:
### run this cell

BLOCK_SIZE = 16

SYSTEM_A = list(rng.integers(0, 50000, size=96))   # 6 blocks
SYSTEM_B = list(rng.integers(0, 50000, size=96))   # a different one

def request(system, question_len=40):
  return system + list(rng.integers(0, 50000, size=question_len))

print(f'system prompts are {len(SYSTEM_A)//BLOCK_SIZE} blocks each')

# Exercise 1: hash the blocks

One hash per full block, so two requests that begin the same way produce the
same hashes for as far as they agree.

In [ ]:
def block_hashes(tokens, block_size=BLOCK_SIZE):
  """One hash per FULL block. Each hash covers every token up to and
  including that block, because block k's K and V depend on blocks 0..k."""
  hashes, block_hash = [], hashlib.sha256()
  for start in range(0, len(tokens) - block_size + 1, block_size):
    block_hash = block_hash.copy()
    block_hash.update(np.array(tokens[start:start+block_size], dtype=np.int64).tobytes())
    hashes.append(block_hash.hexdigest()[:16])
  return hashes

first_a = block_hashes(request(SYSTEM_A))
second_a = block_hashes(request(SYSTEM_A))
first_b = block_hashes(request(SYSTEM_B))

print('two requests with system A share', sum(hash_one==hash_two for hash_one,hash_two in zip(first_a,second_a)), 'block hashes')
print('A against B share              ', sum(hash_one==hash_two for hash_one,hash_two in zip(first_a,first_b)))

# Exercise 2: the trap

Take the same sixteen tokens and put them after two different system prompts.
Do they belong in the same cache entry?

Think about what a KV block physically contains before you answer.

In [ ]:
shared_tail = list(rng.integers(0, 50000, size=BLOCK_SIZE))

one = SYSTEM_A + shared_tail        # tail follows system A
two = SYSTEM_B + shared_tail        # the SAME tail, following system B

hashes_after_a, hashes_after_b = block_hashes(one), block_hashes(two)

print('the two requests end with identical tokens:', one[-BLOCK_SIZE:] == two[-BLOCK_SIZE:])
print('and the hash of that last block matches:   ', hashes_after_a[-1] == hashes_after_b[-1])
print('\nIt must NOT match. The K and V of those tokens depend on everything')
print('before them, and the two prefixes are different.')

# Exercise 3: the cache

A hash to block-id map, an LRU eviction order, and a lookup that returns the
longest cached **prefix**.

In [ ]:
class PrefixCache:
  def __init__(self, capacity_blocks):
    self.capacity_blocks = capacity_blocks
    self.cache = OrderedDict()          # hash -> physical block id
    self.hits = self.misses = 0
    self.next_block = 0

  def lookup(self, hashes):
    """Return the block ids for the longest cached PREFIX of `hashes`."""
    found_blocks = []
    for block_hash in hashes:
      if block_hash not in self.cache:
        break                            # a gap ends the prefix
      self.cache.move_to_end(block_hash)
      found_blocks.append(self.cache[block_hash])
    self.hits += len(found_blocks)
    self.misses += len(hashes) - len(found_blocks)
    return found_blocks

  def insert(self, hashes):
    for block_hash in hashes:
      if block_hash in self.cache:
        self.cache.move_to_end(block_hash)
        continue
      self.cache[block_hash] = self.next_block
      self.next_block += 1
      if len(self.cache) > self.capacity_blocks:
        self.cache.popitem(last=False)   # least recently used

prefix_cache = PrefixCache(64)
first_request = block_hashes(request(SYSTEM_A))
prefix_cache.insert(first_request)
second_request = block_hashes(request(SYSTEM_A))
print(f'second request hit {len(prefix_cache.lookup(second_request))} of {len(second_request)} blocks')

# Exercise 4: run a workload through it

Five popular system prompts and a long tail of unique ones, which is roughly
what a real deployment looks like. Sweep the capacity.

In [ ]:
systems = [list(rng.integers(0, 50000, size=96)) for _ in range(5)]
weights = np.array([.4,.2,.15,.15,.1])

def trace(num_requests=3000, popular_frac=0.8):
  stream_hashes = []
  for _ in range(num_requests):
    if rng.random() < popular_frac:
      system_prompt = systems[rng.choice(len(systems), p=weights)]
    else:
      system_prompt = list(rng.integers(0, 50000, size=96))    # a unique prompt
    stream_hashes.append(block_hashes(request(system_prompt)))
  return stream_hashes

stream = trace()
print(f"{'blocks of cache':>16} {'hit rate':>9} {'system prompts held':>21}")
for capacity_blocks in (6, 12, 30, 60, 120, 600):
  prefix_cache = PrefixCache(capacity_blocks)
  for request_hashes in stream:
    prefix_cache.lookup(request_hashes)
    prefix_cache.insert(request_hashes)
  rate = prefix_cache.hits/(prefix_cache.hits+prefix_cache.misses)
  print(f'{capacity_blocks:>16} {100*rate:>8.1f}% {capacity_blocks/6:>20.0f}')

### The three things that make this correct

**The hash chains.** The hash of block `k` covers token 0 through the last
token of block `k`. It does not cover only the sixteen tokens of block `k`.

Hash the block alone, and Exercise 2 gives two different requests the same
cache entry. That entry returns K and V that the engine computed against the
wrong prefix. The output stays fluent and becomes wrong. No failure mode is
worse.

**The lookup stops at the first gap.** You cannot use block 4 without block 3,
because block 4 has nothing to attend against. A cache that returns a set, and
not a prefix, reports a beautiful hit rate and makes nonsense.

**The cache hashes only full blocks.** The engine still writes into a partial
block. Its contents are not final. A hash of it caches a value that changes
one token later.

### And the number

The hit rate saturates almost at once. Five system prompts carry most of the
traffic. So a cache with room for a few of them holds almost all of what a
cache one hundred times larger holds.

That gives the useful conclusion. Prefix caching does not need much memory. It
is cheap. This is why teams find that they can afford a much longer system
prompt than they planned for.

    ./vc guide 9